In [1]:
import requests
import pandas as pd
import time
import os

# --- 1. CONFIGURATION ---
save_folder = r"C:\Users\asus zb\OneDrive\Desktop\Extra projects"
file_name = "Beijing_ALL_12_Sites_Met_Data.csv"
full_save_path = os.path.join(save_folder, file_name)

# EXACT COORDINATES for all 12 UCI Stations
locations = {
    "Aotizhongxin":  {"lat": 39.982, "lon": 116.417},
    "Changping":     {"lat": 40.217, "lon": 116.230},
    "Dingling":      {"lat": 40.292, "lon": 116.220},
    "Dongsi":        {"lat": 39.929, "lon": 116.417},
    "Guanyuan":      {"lat": 39.929, "lon": 116.339},
    "Gucheng":       {"lat": 39.914, "lon": 116.184},
    "Huairou":       {"lat": 40.328, "lon": 116.628},
    "Nongzhanguan":  {"lat": 39.937, "lon": 116.461},
    "Shunyi":        {"lat": 40.127, "lon": 116.655},
    "Tiantan":       {"lat": 39.886, "lon": 116.407},
    "Wanliu":        {"lat": 39.987, "lon": 116.287},
    "Wanshouxigong": {"lat": 39.878, "lon": 116.352}
}

# Matching the UCI Dataset dates exactly
start_date = "20130301"
end_date = "20170228"

parameters = "ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN" # We only need Solar, as UCI has Temp/Wind
base_url = "https://power.larc.nasa.gov/api/temporal/hourly/point"

# --- 2. FETCH ENGINE ---
print(f"🚀 Starting Batch Extraction for 12 Sites ({start_date}-{end_date})...")
all_data = []

for city, coords in locations.items():
    print(f"   📡 Fetching: {city}...")
    params = {
        "parameters": parameters,
        "community": "RE",
        "longitude": coords['lon'],
        "latitude": coords['lat'],
        "start": start_date,
        "end": end_date,
        "format": "JSON"
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        if response.status_code == 200:
            data = response.json()
            df = pd.DataFrame(data['properties']['parameter'])
            df.index = pd.to_datetime(df.index, format='%Y%m%d%H')
            df.index.name = 'Datetime_UTC'
            df['Station'] = city
            all_data.append(df)
        else:
            print(f"   ❌ Error: {response.status_code}")
    except Exception as e:
        print(f"   ❌ Connection Failed: {e}")
    
    time.sleep(1.0) # Fast pause

# --- 3. SAVE ---
if all_data:
    final_df = pd.concat(all_data)
    final_df.rename(columns={
        'ALLSKY_SFC_SW_DWN': 'GHI',
        'CLRSKY_SFC_SW_DWN': 'Clear_Sky_GHI'
    }, inplace=True)
    
    final_df.to_csv(full_save_path)
    print(f"\n✅ 12-STATION SOLAR DATA SAVED!\nLocation: {full_save_path}")

🚀 Starting Batch Extraction for 12 Sites (20130301-20170228)...
   📡 Fetching: Aotizhongxin...
   📡 Fetching: Changping...
   📡 Fetching: Dingling...
   📡 Fetching: Dongsi...
   📡 Fetching: Guanyuan...
   📡 Fetching: Gucheng...
   📡 Fetching: Huairou...
   📡 Fetching: Nongzhanguan...
   📡 Fetching: Shunyi...
   📡 Fetching: Tiantan...
   📡 Fetching: Wanliu...
   📡 Fetching: Wanshouxigong...

✅ 12-STATION SOLAR DATA SAVED!
Location: C:\Users\asus zb\OneDrive\Desktop\Extra projects\Beijing_ALL_12_Sites_Met_Data.csv


In [2]:
import pandas as pd
import os
import glob

# --- CONFIGURATION ---
# 1. Your specific folder with the 12 pollution files
pollution_folder = r"C:\Users\asus zb\OneDrive\Desktop\Extra projects\PRSA_Data_20130301-20170228"

# 2. The NASA Solar Data file (Generated in the previous step)
# I assume this is saved in your main 'Extra projects' folder
nasa_file = r"C:\Users\asus zb\OneDrive\Desktop\Extra projects\Beijing_ALL_12_Sites_Met_Data.csv"

# 3. Where to save the final result
output_file = r"C:\Users\asus zb\OneDrive\Desktop\Extra projects\FINAL_REGIONAL_DATASET.csv"


# --- 1. LOAD ALL 12 POLLUTION FILES ---
print(f"📂 Scanning folder: {pollution_folder}...")
all_files = glob.glob(os.path.join(pollution_folder, "*.csv"))

if not all_files:
    print("❌ ERROR: No CSV files found in the folder!")
    print("   Check if the path is correct or if the folder is empty.")
else:
    print(f"   ✅ Found {len(all_files)} files. Merging them now...")

    df_list = []
    for filename in all_files:
        df = pd.read_csv(filename)
        df_list.append(df)

    # Combine into one big dataframe
    df_pol = pd.concat(df_list, ignore_index=True)
    
    # Create a proper Datetime column
    df_pol['Datetime'] = pd.to_datetime(df_pol[['year', 'month', 'day', 'hour']])
    
    print(f"   ✅ Loaded Pollution Data: {len(df_pol)} rows.")


    # --- 2. LOAD NASA SOLAR DATA ---
    if os.path.exists(nasa_file):
        print("☀️ Loading NASA Solar Data...")
        df_nasa = pd.read_csv(nasa_file)
        df_nasa['Datetime_UTC'] = pd.to_datetime(df_nasa['Datetime_UTC'])

        # TIMEZONE SHIFT (UTC -> Beijing Time +8)
        df_nasa['Datetime'] = df_nasa['Datetime_UTC'] + pd.Timedelta(hours=8)
        
        # --- 3. MERGE DATASETS ---
        print("🔗 Linking Pollution & Solar Data (Matching by Time & Station)...")
        
        # Merge on BOTH 'Datetime' and 'Station'
        # Note: The UCI file uses 'station' (lowercase), NASA uses 'Station' (Title case)
        # We normalize them to be safe
        df_pol['station'] = df_pol['station'].str.strip()
        df_nasa['Station'] = df_nasa['Station'].str.strip()
        
        df_final = pd.merge(df_pol, df_nasa, 
                            left_on=['Datetime', 'station'], 
                            right_on=['Datetime', 'Station'], 
                            how='inner')

        # --- 4. CLEANUP & SAVE ---
        # Select useful columns
        cols = ['Datetime', 'station', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 
                'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'GHI', 'Clear_Sky_GHI']
        
        # Filter for only existing columns (in case some are missing)
        valid_cols = [c for c in cols if c in df_final.columns]
        df_final = df_final[valid_cols]

        # Handle missing values (Interpolation)
        df_final = df_final.interpolate(method='linear', limit_direction='both')
        df_final.dropna(inplace=True)

        # Save
        df_final.to_csv(output_file, index=False)
        
        print("\n" + "="*50)
        print(f"🎉 FINAL DATASET READY!")
        print(f"📍 Location: {output_file}")
        print(f"📊 Total Rows: {len(df_final)}")
        print(f"📅 Stations Covered: {df_final['station'].unique()}")
        print("="*50)
        
        # Show a sample
        print(df_final.head())
        
    else:
        print(f"❌ ERROR: Could not find the NASA file at: {nasa_file}")
        print("   Did you run the previous script to fetch the solar data?")

📂 Scanning folder: C:\Users\asus zb\OneDrive\Desktop\Extra projects\PRSA_Data_20130301-20170228...
   ✅ Found 12 files. Merging them now...
   ✅ Loaded Pollution Data: 420768 rows.
☀️ Loading NASA Solar Data...
🔗 Linking Pollution & Solar Data (Matching by Time & Station)...


C:\Users\asus zb\AppData\Local\Temp\ipykernel_21492\1891832833.py:74: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_final = df_final.interpolate(method='linear', limit_direction='both')



🎉 FINAL DATASET READY!
📍 Location: C:\Users\asus zb\OneDrive\Desktop\Extra projects\FINAL_REGIONAL_DATASET.csv
📊 Total Rows: 418850
📅 Stations Covered: ['Aotizhongxin' 'Changping' 'Dingling' 'Dongsi' 'Guanyuan' 'Gucheng'
 'Huairou' 'Nongzhanguan' 'Shunyi' 'Tiantan' 'Wanliu' 'Wanshouxigong']
             Datetime       station  PM2.5  PM10   SO2   NO2     CO    O3  \
0 2013-03-01 08:00:00  Aotizhongxin    3.0   6.0  16.0  43.0  500.0  45.0   
1 2013-03-01 09:00:00  Aotizhongxin    3.0   8.0  12.0  28.0  400.0  59.0   
2 2013-03-01 10:00:00  Aotizhongxin    3.0   6.0   9.0  12.0  400.0  72.0   
3 2013-03-01 11:00:00  Aotizhongxin    3.0   6.0   9.0  14.0  400.0  71.0   
4 2013-03-01 12:00:00  Aotizhongxin    3.0   6.0   7.0  13.0  300.0  74.0   

   TEMP    PRES  DEWP  RAIN   wd  WSPM  GHI  Clear_Sky_GHI  
0   0.1  1028.3 -19.2   0.0  NNW   4.1  0.0            0.0  
1   1.2  1028.5 -19.3   0.0    N   2.6  0.0            0.0  
2   1.9  1028.2 -19.4   0.0  NNW   3.6  0.0            0.0  
